# WhisperWatch production backend in Colab

This notebook loads the exact original boosted model, verifies its scores on real saved Whisper outputs, and runs the integrated backend on short English recordings. It is the production demo entry point. The separate `run_colab.ipynb` contains the historical training workflow.

Open this file in [Google Colab](https://colab.research.google.com/). A CPU runtime is enough for Steps 1–3. For Step 4, choose **Runtime → Change runtime type → T4 GPU** before starting the notebook. This keeps all uploads and verification in the same hosted runtime.

## 1. Install backend dependencies

The package versions for scikit-learn and Whisper match the saved artifact and previous Colab live runs. If Colab explicitly requests a Python session restart after installing packages, restart before Step 2.

In [ ]:
%pip install -q "scikit-learn==1.8.0" "joblib==1.5.3" "pandas==2.2.3" "faster-whisper==1.2.1" "ctranslate2==4.9.0" "av==16.0.1" "huggingface-hub==1.32.0"

## 2. Upload the code and approved weights

Upload `whisperwatch-backend-code.zip` when prompted. This is a local copy of the backend code, so it works with a private GitHub repo without a token. Then upload the original `whisperwatch-boosted-results.zip`. The loader checks the exact expected SHA-256 before reading the model and verifies the ten-feature schema.

In [ ]:
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

project = Path("/content/whisperwatch_backend")
project.mkdir(exist_ok=True)

def upload_one(prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one ZIP file.")
    return BytesIO(next(iter(uploaded.values())))

modules = ("__init__", "asr", "features", "inference", "ranking", "colab_backend", "backend_smoke")
members = [f"src/{module}.py" for module in modules] + ["notebooks/fixtures/backend_smoke.json"]
with ZipFile(upload_one("Upload whisperwatch-backend-code.zip.")) as bundle:
    for name in members:
        path = project / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(bundle.read(name))

sys.path.insert(0, str(project))
from src.features import FEATURE_NAMES
from src.inference import install_production_model, load_production_model

path = install_production_model(upload_one("Upload whisperwatch-boosted-results.zip."))
artifact = load_production_model()
assert tuple(artifact["feature_names"]) == FEATURE_NAMES
print("Verified frozen classifier:", path)
print("Ordered model inputs:", ", ".join(FEATURE_NAMES))
print("CPU classifier is ready.")
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
print(gpu.stdout.strip() if gpu and gpu.returncode == 0 and gpu.stdout.strip() else "No Colab GPU detected.")
try:
    print("Gradio installed:", version("gradio"))
except PackageNotFoundError:
    print("Gradio can be installed with the frontend after integration.")


## 3. Verify and rank genuine saved examples

This cell re-extracts features from ten real LibriSpeech Whisper segment records, verifies their scores against the original Colab export, and ranks the clips. They are examples for checking software behavior. The ranking over these ten examples is specific to this batch.

In [ ]:
import json
import math
from IPython.display import display
import pandas as pd
from src.features import extract_features
from src.inference import predict_from_features, rank_recordings

fixture = json.loads((project / "notebooks/fixtures/backend_smoke.json").read_text(encoding="utf-8"))
analyzed = []
for clip in fixture["clips"]:
    features = extract_features(clip["segments"], clip["transcript"], clip["duration_s"])
    result = predict_from_features(features)
    assert math.isclose(result["review_score"], clip["expected_score"], rel_tol=1e-10, abs_tol=1e-12)
    analyzed.append(dict(result, clip_id=clip["clip_id"], transcript=clip["transcript"]))
display(analyzed[2])
display(pd.DataFrame(rank_recordings(analyzed)))
subprocess.run([sys.executable, "-m", "src.backend_smoke"], cwd=project, check=True)


## 4. Run Whisper Large V3 on real audio

Use a hosted GPU runtime. Install GPU libraries in the first cell, then run the second cell and upload one to five English clips of at most 30 seconds each. A single upload returns a classifier score. Upload five clips to see one selected review slot. A fresh runtime downloads about 3.1 GB of pretrained Whisper weights on first use. The live code uses the same pinned Large V3 conversion, decoding settings, and input feature calculations used in the earlier live evaluations.

In [ ]:
if not gpu or gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("Step 4 needs a hosted GPU. Select T4 GPU, then rerun Steps 1–3 in that runtime.")
%pip install -q "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"


In [ ]:
from src.colab_backend import gpu_process_env

print("Choose one to five short English recordings.")
uploads = files.upload()
if not 1 <= len(uploads) <= 5:
    raise ValueError("Choose one to five audio files.")
audio_dir = project / "uploads"
audio_dir.mkdir(exist_ok=True)
paths = []
for i, (filename, content) in enumerate(uploads.items(), start=1):
    path = audio_dir / f"{i:02d}-{Path(filename).name}"
    path.write_bytes(content)
    paths.append(str(path))
live_output = project / "live_backend_response.json"
subprocess.run([sys.executable, "-m", "src.colab_backend", str(live_output), *paths],
               cwd=project, env=gpu_process_env(), check=True)
response = json.loads(live_output.read_text(encoding="utf-8"))
display(response["analysis"])
if response["batch"]:
    display(pd.DataFrame(response["batch"]))


## 5. Prepare the frontend environment

Member B can call `src.asr.analyze_audio` and `src.inference.rank_recordings` in a Python process that has the CUDA environment from `src.colab_backend.gpu_process_env()`. The deployed app should run from the project root and reuse the same weights. See `docs/BACKEND_HANDOFF.md` for the exact callback examples and output contract.